# Explore ESM-2 embeddings — interactive PCA + full-D neighbours
CPU-only; reads the precomputed `embeddings/esm2_*.h5` + catalog. Interactive PCA (hover = identity) is the *map*; **`neighbors()` in full 1280-D is the rigorous per-point tool** — PCA is lossy and PC1≈length here, so trust neighbours over 2-D position.

Needs `plotly` (`pip install plotly`).

In [ ]:
import sys, os
for p in ['.', '..']:
    if os.path.isdir(os.path.join(p, 'embed_lib')): sys.path.insert(0, p); break
from embed_lib import explore
import numpy as np, pandas as pd

MODEL, LAYER, POOL = '650M', None, 'mean'   # LAYER=None -> deepest stored; try 6 for the early-peak layer
meta, E, ids = explore.load_embeddings(MODEL, LAYER, POOL)
coords, ev = explore.pca2d(E)
print(len(ids), 'proteins,', E.shape[1], 'dims, layer', int(meta['_layer'].iloc[0]))
meta['class'].value_counts()

## Interactive PCA — hover any point to see its identity

In [ ]:
fig = explore.make_figure(coords, meta, color='class',
    title=f"PCA {MODEL} L{int(meta['_layer'].iloc[0])} (PC1 {ev[0]:.1f}%, PC2 {ev[1]:.1f}%) — hover to identify")
fig.show()

## Recolour: length confound (PC1≈length) or any catalog column

In [ ]:
explore.make_figure(coords, meta.assign(log_length=np.log10(meta['length'])),
    color='log_length', title='PCA coloured by log10(length)').show()

## Spotlight specific ORFs on the fly (no need to edit the catalog)

In [ ]:
spotlight = ['c12riboseqorf747']   # <-- your ORF ids (releasev45_id)
m2 = meta.copy(); m2['highlight'] = m2.index.isin(spotlight)
explore.make_figure(coords, m2, color='class', title='spotlight').show()

## Full-D nearest neighbours of a point — the real analysis
Who is this protein actually closest to in 1280-D? Restrict to metal binders to ask *'is it near metal-binding proteins, and which ion?'*

In [ ]:
QUERY = 'c12riboseqorf747'   # any protein_id (microprotein, metal binder, ...)
explore.neighbors(QUERY, ids, E, meta, k=15, restrict='metal')

In [ ]:
# quick verdict: of its nearest metal binders, which ions dominate + how close
prof, med = explore.metal_ion_profile(QUERY, ids, E, meta, k=15)
print('nearest-metal ion tally:', prof, '| median cos-dist', round(med, 3))
print()
print(meta.loc[QUERY, ['class','length','orf_type','gene_name']].to_dict())
print(meta.loc[QUERY, 'seq'])

## Export a standalone interactive HTML (open in any browser / share)

In [ ]:
explore.write_html(fig, '../results/interactive/pca_explore.html')
print('wrote results/interactive/pca_explore.html')